# Level 10: Quantum Error Correction
## Q# Edition

In this notebook, you will:

1. Implement the **3-qubit bit-flip code** in Q#
2. Simulate errors and **syndrome-based correction**
3. Test correction over many trials
4. Understand fault-tolerant quantum computing concepts

In [ ]:
import qsharp
from qsharp_widgets import Circuit
print("Q# ready!")

---
## 1. Encoding a Logical Qubit

$$|0_L\rangle = |000\rangle, \quad |1_L\rangle = |111\rangle$$

Encoding is done with CNOT gates:

In [ ]:
%%qsharp

open Microsoft.Quantum.Diagnostics;

// Encode a single qubit into 3-qubit bit-flip code
operation Encode(data : Qubit, anc1 : Qubit, anc2 : Qubit) : Unit {
    CNOT(data, anc1);
    CNOT(data, anc2);
}

// Decode back to single qubit
operation Decode(data : Qubit, anc1 : Qubit, anc2 : Qubit) : Unit {
    CNOT(data, anc2);
    CNOT(data, anc1);
}

// Show encoding
operation ShowEncoding() : Unit {
    use qs = Qubit[3];
    
    // Prepare |+> on first qubit
    H(qs[0]);
    Message("Before encoding:");
    DumpMachine();
    
    // Encode
    Encode(qs[0], qs[1], qs[2]);
    Message("\nAfter encoding (|+_L> = |000> + |111>)/√2:");
    DumpMachine();
    
    ResetAll(qs);
}

ShowEncoding()

---
## 2. Error Detection and Correction

In [ ]:
%%qsharp

// Detect bit-flip error using syndrome measurement
// Returns (s1, s2) syndrome bits:
//   (Zero, Zero) → no error
//   (One,  Zero) → error on qubit 1 (anc1)
//   (Zero, One)  → error on qubit 2 (anc2)
//   (One,  One)  → error on qubit 0 (data)
operation MeasureSyndrome(data : Qubit, anc1 : Qubit, anc2 : Qubit) : (Result, Result) {
    use syn = Qubit[2];
    
    // Compare data with anc1
    CNOT(data, syn[0]);
    CNOT(anc1, syn[0]);
    
    // Compare data with anc2
    CNOT(data, syn[1]);
    CNOT(anc2, syn[1]);
    
    let s1 = M(syn[0]);
    let s2 = M(syn[1]);
    
    ResetAll(syn);
    return (s1, s2);
}

// Correct based on syndrome
operation CorrectBitFlip(data : Qubit, anc1 : Qubit, anc2 : Qubit, s1 : Result, s2 : Result) : Unit {
    if s1 == One and s2 == One {
        // Error on data qubit
        X(data);
    } elif s1 == One and s2 == Zero {
        // Error on anc1
        X(anc1);
    } elif s1 == Zero and s2 == One {
        // Error on anc2
        X(anc2);
    }
    // s1 == Zero, s2 == Zero → no error, do nothing
}

In [ ]:
%%qsharp

// Full error correction demo: encode → error → detect → correct → decode → verify
operation BitFlipCorrection(errorQubit : Int) : Bool {
    use qs = Qubit[3];
    
    // Prepare |1> as our logical state
    X(qs[0]);
    
    // Encode
    Encode(qs[0], qs[1], qs[2]);
    
    // Introduce bit-flip error
    if errorQubit == 0 { X(qs[0]); }
    elif errorQubit == 1 { X(qs[1]); }
    elif errorQubit == 2 { X(qs[2]); }
    // errorQubit == -1 means no error
    
    // Detect
    let (s1, s2) = MeasureSyndrome(qs[0], qs[1], qs[2]);
    
    // Correct
    CorrectBitFlip(qs[0], qs[1], qs[2], s1, s2);
    
    // Decode
    Decode(qs[0], qs[1], qs[2]);
    
    // Verify: should be |1>
    let result = M(qs[0]);
    ResetAll(qs);
    
    return result == One;  // true = correctly recovered
}

// Test all scenarios
Message("=== Bit-Flip Error Correction ===");
Message("");

let noError = BitFlipCorrection(-1);
Message($"No error:        recovered = {noError}");

let err0 = BitFlipCorrection(0);
Message($"Error on qubit 0: recovered = {err0}");

let err1 = BitFlipCorrection(1);
Message($"Error on qubit 1: recovered = {err1}");

let err2 = BitFlipCorrection(2);
Message($"Error on qubit 2: recovered = {err2}");

---
## 3. Statistical Verification

Let's run many trials to confirm the code works reliably with superposition states.

In [ ]:
%%qsharp

// Test with superposition: encode |+>, introduce error, correct, verify
operation BitFlipSuperpositionTest(errorQubit : Int, shots : Int) : Int {
    mutable correct = 0;
    
    for _ in 0..shots - 1 {
        use qs = Qubit[3];
        
        // Prepare |+>
        H(qs[0]);
        
        // Encode
        Encode(qs[0], qs[1], qs[2]);
        
        // Error
        if errorQubit == 0 { X(qs[0]); }
        elif errorQubit == 1 { X(qs[1]); }
        elif errorQubit == 2 { X(qs[2]); }
        
        // Detect and correct
        let (s1, s2) = MeasureSyndrome(qs[0], qs[1], qs[2]);
        CorrectBitFlip(qs[0], qs[1], qs[2], s1, s2);
        
        // Decode
        Decode(qs[0], qs[1], qs[2]);
        
        // Undo H: if correction worked, should return to |0>
        H(qs[0]);
        let result = M(qs[0]);
        if result == Zero {
            set correct += 1;
        }
        
        ResetAll(qs);
    }
    
    return correct;
}

In [ ]:
import matplotlib.pyplot as plt

shots = 2000
scenarios = [-1, 0, 1, 2]
labels = ['No Error', 'Error Q0', 'Error Q1', 'Error Q2']
correct_rates = []

for eq in scenarios:
    correct = qsharp.eval(f"BitFlipSuperpositionTest({eq}, {shots})")
    rate = correct / shots * 100
    correct_rates.append(rate)
    print(f"{labels[scenarios.index(eq)]}: {rate:.1f}% correct")

plt.figure(figsize=(10, 5))
bars = plt.bar(labels, correct_rates, color=['green', 'steelblue', 'steelblue', 'steelblue'])
plt.ylabel('Correct Recovery Rate (%)')
plt.title('3-Qubit Bit-Flip Code: Correction Success Rate')
plt.ylim(90, 101)
for bar, rate in zip(bars, correct_rates):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
             f'{rate:.1f}%', ha='center', fontsize=12)
plt.show()

---
## 4. Phase-Flip Code in Q#

In [ ]:
%%qsharp

// Phase-flip code: use Hadamard basis
operation PhaseFlipCorrection(errorQubit : Int) : Bool {
    use qs = Qubit[3];
    
    // Prepare |1>
    X(qs[0]);
    
    // Encode for phase-flip
    Encode(qs[0], qs[1], qs[2]);
    H(qs[0]); H(qs[1]); H(qs[2]);
    
    // Phase-flip error (Z gate)
    if errorQubit == 0 { Z(qs[0]); }
    elif errorQubit == 1 { Z(qs[1]); }
    elif errorQubit == 2 { Z(qs[2]); }
    
    // Switch back to Z-basis
    H(qs[0]); H(qs[1]); H(qs[2]);
    
    // Now it looks like a bit-flip — detect and correct
    let (s1, s2) = MeasureSyndrome(qs[0], qs[1], qs[2]);
    CorrectBitFlip(qs[0], qs[1], qs[2], s1, s2);
    
    // Decode
    Decode(qs[0], qs[1], qs[2]);
    
    let result = M(qs[0]);
    ResetAll(qs);
    return result == One;
}

Message("=== Phase-Flip Error Correction ===");
Message("");
Message($"No error:        recovered = {PhaseFlipCorrection(-1)}");
Message($"Phase error Q0:  recovered = {PhaseFlipCorrection(0)}");
Message($"Phase error Q1:  recovered = {PhaseFlipCorrection(1)}");
Message($"Phase error Q2:  recovered = {PhaseFlipCorrection(2)}");

---
## 5. The Quantum Error Correction Landscape

| Code | Errors Corrected | Qubits | Key Idea |
|------|-----------------|--------|----------|
| Bit-flip (3,1) | 1 X error | 3 | Majority vote in Z-basis |
| Phase-flip (3,1) | 1 Z error | 3 | Majority vote in X-basis |
| Shor [[9,1,3]] | 1 arbitrary | 9 | Concatenate both |
| Steane [[7,1,3]] | 1 arbitrary | 7 | CSS code |
| Surface code | Many | ~1000/logical | 2D lattice, high threshold |

---
## 6. Key Takeaways

| Concept | Description |
|---------|-------------|
| **Encoding** | Spread 1 logical qubit across 3 physical |
| **Syndrome** | Measure ancillas to find error location |
| **Correction** | Apply X/Z to fix without measuring data |
| **No-cloning** | Encoding is NOT copying — it entangles |
| **Threshold** | QEC only works below a physical error rate |

---
## 7. Final Challenges

1. **Two errors**: Run `BitFlipCorrection` with X applied to 2 qubits. What happens?
2. **Shor code**: Implement a 9-qubit code using 3 groups of 3 (outer phase-flip, inner bit-flip)
3. **CSS code**: Research the 7-qubit Steane code and implement syndrome measurement

### Congratulations!
You've completed the Introduction to Quantum Computing course!

**Your quantum journey:**
- Levels 1-2: Qubits, gates, and measurement
- Levels 3-4: Entanglement and quantum protocols
- Levels 5-8: Quantum algorithms (Deutsch-Jozsa → Shor's)
- Levels 9-10: NISQ computing and error correction

The quantum future is being built right now — and you have the foundations to be part of it!

In [ ]:
%%qsharp

// Your challenge code here!

---
**Back to: [Course README](../README.md)**